# 02 · Rule experiments: NYC Yellow Taxi

Phase 2, Step 2.6. Reads the latest rule run from `data/dataguard.duckdb` and shows how the rules
in `configs/quality_rules.yml` behave on the real data: rows quarantined and warned per month,
failures per rule, the comparison with the profiling findings, and example rows with their reasons.

**Before running:**
```
python src/ingestion/ingest_data.py
python src/quality/run_rules.py
```

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from common import db
from common.config import load_datasets_config
from ingestion.bronze_loader import current_files, read_bronze_sql
from quality import row_rules
from quality.rules import load_rules
from quality.standardize import standardize_file

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Settings

In [ ]:
config = load_datasets_config()
dataset = config.dataset("yellow_taxi")
DATABASE = Path(os.environ.get("DATAGUARD_DATABASE", config.paths["duckdb"]))
LOOKUPS = Path(os.environ.get("DATAGUARD_LOOKUPS", config.paths["incoming"]))
EXAMPLE_MONTH = os.environ.get("DATAGUARD_EXAMPLE_MONTH")  # default: first month in Bronze

con = db.connect(DATABASE, read_only=True)
run_id = con.execute(
    "SELECT run_id FROM rule_run_summary ORDER BY run_id DESC LIMIT 1"
).fetchone()
assert run_id, "no rule run found: run python src/quality/run_rules.py first"
run_id = run_id[0]
print("latest rule run:", run_id)

## 1 · Rows quarantined and warned per month

A row is **quarantined** when it fails at least one quarantine rule and **warned** when it fails
only warn rules.

In [ ]:
summary = con.execute(
    """
    SELECT period, rows_checked,
           rows_quarantined, round(100.0 * rows_quarantined / rows_checked, 3) AS quarantined_pct,
           rows_warned, round(100.0 * rows_warned / rows_checked, 3) AS warned_pct,
           standardization
    FROM rule_run_summary WHERE run_id = ? ORDER BY period
    """,
    [run_id],
).df()
summary

In [ ]:
fig = px.bar(
    summary.melt(id_vars="period", value_vars=["quarantined_pct", "warned_pct"],
                 var_name="outcome", value_name="pct"),
    x="period", y="pct", color="outcome", barmode="group",
    title="Share of rows quarantined and warned per month (%)",
)
fig.show()

## 2 · Failures per rule

In [ ]:
per_rule = con.execute(
    """
    SELECT rule_id, category, action, severity,
           sum(rows_failed) AS rows_failed,
           round(100.0 * sum(rows_failed) / sum(rows_checked), 4) AS pct_all_months,
           round(100.0 * max(rows_failed / rows_checked), 4) AS worst_month_pct,
           count_if(rows_failed > 0) AS months_affected
    FROM rule_run_log WHERE run_id = ?
    GROUP BY ALL ORDER BY rows_failed DESC
    """,
    [run_id],
).df()
per_rule

## 3 · Comparison with the profiling findings

Worst-month percentages measured in Phase 0 (`docs/data_profile.md`). Small differences are
expected where a rule is defined slightly differently from the profiling check (for example
NULL values pass value rules). A large difference means the rule needs a second look.

In [ ]:
profiled = pd.DataFrame(
    [
        ("trip_metadata_missing", 5.33), ("distance_not_zero", 3.4618),
        ("passengers_not_zero", 1.7319), ("fare_not_negative", 1.2629),
        ("total_not_negative", 1.1789), ("trip_has_duration", 0.0851),
        ("fare_not_zero", 0.0362), ("dropoff_not_before_pickup", 0.0265),
        ("pickup_in_file_month", 0.0070), ("distance_max_100_miles", 0.0048),
        ("passengers_max_6", 0.0022), ("trip_under_24_hours", 0.0008),
        ("pickup_zone_known", 0.0), ("dropoff_zone_known", 0.0),
    ],
    columns=["rule_id", "profiled_worst_month_pct"],
)
comparison = per_rule[["rule_id", "worst_month_pct"]].merge(profiled, on="rule_id", how="left")
comparison["difference"] = (comparison["worst_month_pct"] - comparison["profiled_worst_month_pct"]).round(4)
comparison

## 4 · Example rows per rule

Up to five failing rows per rule for one month, with the reasons stored for them.

In [ ]:
files = current_files(con, dataset)
month = EXAMPLE_MONTH or next(iter(files))
example_con = db.connect(DATABASE, read_only=True)
row_rules.load_lookups(example_con, dataset, LOOKUPS)
sql, _ = standardize_file(example_con, files[month], dataset)
evaluated = row_rules.evaluate_sql(example_con, load_rules(dataset), f"({sql})", period=month)
examples = example_con.execute(
    f"""
    SELECT rule_id, bronze_row_id, reason
    FROM (
        SELECT bronze_row_id, unnest(failed_rules) AS rule_id, unnest(reasons) AS reason
        FROM ({evaluated})
    )
    QUALIFY row_number() OVER (PARTITION BY rule_id ORDER BY bronze_row_id) <= 5
    ORDER BY rule_id, bronze_row_id
    """
).df()
print("month:", month)
examples

## 5 · Rows failing more than one rule

Shows which rules fire together. Negative fares, for example, usually come with a negative total.

In [ ]:
combinations = example_con.execute(
    f"""
    SELECT array_to_string(failed_rules, ' + ') AS rules, count(*) AS rows
    FROM ({evaluated}) WHERE len(failed_rules) > 1
    GROUP BY ALL ORDER BY rows DESC LIMIT 15
    """
).df()
combinations

In [ ]:
example_con.close()
con.close()